# Scene Selection & Data Acquisition (2016–2021)
### Purpose
This notebook implements the data acquisition and scene selection stage of the surface water analysis pipeline. It searches satellite imagery from Sentinel-2 and Landsat 8/9 using the OpenEO API, identifies the best available summer scene per year (2016–2021) within a defined Area of Interest (AOI), and prepares the data for subsequent spectral analysis.

### Study Period and Sensors
- Time span: 2016–2021
- Season: Summer (June–August)
- Sensors: Sentinel-2 Level-2A, Landsat Collection 2 Level-2

### Methodology Overview
For each year and sensor:
1. Connects to the VITO / Terrascope openEO backend
2. Searches Sentinel-2 and Landsat 8/9 summer imagery (June–August) over a user-defined AOI
3. Selects the best scene per year based on lowest cloud cover
4. Exports a CSV summary of selected scenes
5. Downloads raw spectral bands (GeoTIFF) to local disk

### Notes:
- NDWI and analysis are performed in a separate notebook
- Cloud filtering and spatial clipping are handled server-side
- Output GeoTIFFs are analysis-ready

### Outputs
- CSV summary of selected scenes (scene ID, date, cloud cover)
- Downloaded GeoTIFFs for required spectral bands
- Metadata verification output confirming spatial consistency across datasets

In [ ]:
# -----------------------------
# Imports
# -----------------------------
import openeo
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import from_origin
from shapely.geometry import box
from pathlib import Path
import tempfile
import os
import pandas as pd
from tqdm import tqdm
import folium
from folium.features import GeoJson
import csv
from datetime import datetime

#Supress warnings. 
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ---------------------- Configuration ----------------------
# -----------------------------
# PATHS
# -----------------------------
AOI_PATH = "../data/aoi/7205_AOI.geojson"
OUT_DIR = Path("../data/raw")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUT_DIR / "scene_inventory.csv"

# -----------------------------
# YEARS / SEASON
# -----------------------------
YEARS = {
    2016: ("2016-06-01", "2016-08-31"),
    2017: ("2017-06-01", "2017-08-31"),
    2018: ("2018-06-01", "2018-08-31"),
    2019: ("2019-06-01", "2019-08-31"),
    2020: ("2020-06-01", "2020-08-31"),
    2021: ("2021-06-01", "2021-08-31"),
}

# -----------------------------
# SENSOR DEFINITIONS
# -----------------------------
SENSORS = {
    "sentinel2": {
        "collection": "SENTINEL2_L2A",
        "bands": ["B04", "B03", "B02", "B08"],  # Blue, Green, Red, NIR
        "cloud_max": 20
    },
    "landsat": {
        "collection": "LANDSAT8-9_L2",
        "bands": ["B04", "B03", "B02", "B05"],  # RGB + NIR
        "cloud_max": 20
    }
}

sensor_order = list(SENSORS.keys())
for s in SENSORS:
    os.makedirs(Path(OUT_DIR)/s.lower().replace("-", ""), exist_ok=True)

In [ ]:
# -----------------------------
# CONNECT TO OPENEO BACKEND
# -----------------------------
connection = openeo.connect("openeo.vito.be").authenticate_oidc()
print("Authenticated with openEO VITO backend")

In [ ]:
# ---------------------- Load AOI ----------------------
# Load AOI and convert to GeoJSON
aoi_gdf = gpd.read_file(AOI_PATH)
# dissolve in case of multiple polygons
aoi_geom = aoi_gdf.unary_union.__geo_interface__

In [ ]:
# ---------------------- SANITY CHECK ----------------------
# Check if OpenEO returns collections
# s = SENSORS["Sentinel-2"]
# coll = connection.load_collection(
#     s["id"],
#     spatial_extent=aoi_geojson,
#     temporal_extent=["2016-06-01","2016-08-31"],
#     bands=s["rgb_bands"],
#     fetch_metadata=True
# )
# metadata = coll.metadata
# print(metadata)

In [ ]:
# -----------------------------
# Download BATCH Fcn
# -----------------------------
def run_summer_composite(
    connection,
    collection_id,
    bands,
    aoi_geom,
    start,
    end,
    out_dir,
    out_name
):
    cube = connection.load_collection(
        collection_id=collection_id,
        spatial_extent=aoi_geom,
        temporal_extent=[start, end],
        bands=bands
    )

    # Temporal composite (summer median)
    cube = cube.reduce_dimension(
        dimension="t",
        reducer="median"
    )

    cube = cube.save_result(format="GTiff")

    job = cube.create_job(title=out_name)
    job.start_and_wait()

    results = job.get_results()
    results.download_files(out_dir)

In [ ]:
# -----------------------------
# Main download loop
# -----------------------------
inventory = []

for year, (start, end) in YEARS.items():
    print(f"\n=== {year} ({start} → {end}) ===")

    for sensor_name, sensor in SENSORS.items():
        print(f"Downloading {sensor_name} summer composite...")

        sensor_dir = OUT_DIR / sensor_name / str(year)
        sensor_dir.mkdir(parents=True, exist_ok=True)

        out_name = f"{sensor_name}_{year}_summer_raw"

        try:
            run_summer_composite(
                connection=connection,
                collection_id=sensor["collection"],
                bands=sensor["bands"],
                aoi_geom=aoi_geom,
                start=start,
                end=end,
                out_dir=sensor_dir,
                out_name=out_name
            )

            inventory.append({
                "sensor": sensor_name,
                "year": year,
                "start_date": start,
                "end_date": end,
                "bands": ",".join(sensor["bands"]),
                "output_dir": str(sensor_dir),
                "verified": True
            })

        except Exception as e:
            print(f"FAILED: {e}")
            inventory.append({
                "sensor": sensor_name,
                "year": year,
                "start_date": start,
                "end_date": end,
                "bands": ",".join(sensor["bands"]),
                "output_dir": None,
                "verified": False
            })

In [ ]:
# -----------------------------
# CSV Print
# -----------------------------
df = pd.DataFrame(inventory)
df.to_csv(CSV_PATH, index=False)
print(f"\nScene inventory saved to {CSV_PATH}")

In [ ]:
# -----------------------------
# Interactive AOI + Scene Map
# -----------------------------
RAW_DIR = "../data/aoi"

def plot_aoi_interactive_sensors(scene_df, aoi_path, raster_root=RAW_DIR, zoom_start=6):
    gdf = gpd.read_file(aoi_path)
    aoi_geom = gdf.unary_union
    aoi_gdf = gpd.GeoDataFrame(geometry=[aoi_geom], crs=gdf.crs).to_crs(epsg=4326)
    
    # Center map on AOI centroid
    centroid = aoi_gdf.geometry.centroid.iloc[0]
    m = folium.Map(location=[centroid.y, centroid.x], zoom_start=zoom_start)
    
    # Draw AOI
    GeoJson(aoi_gdf.geometry, style_function=lambda x: {"color": "red", "weight": 2}, tooltip="AOI").add_to(m)
    
    color_map = {"sentinel-2": "green", "landsat-8": "blue"}
    
    for _, row in scene_df.iterrows():
        fpath = row["file_path"]
        if fpath and Path(fpath).exists():
            try:
                with rasterio.open(fpath) as ds:
                    geom = box(*ds.bounds)
                    geom_gdf = gpd.GeoDataFrame(geometry=[geom], crs=ds.crs).to_crs(epsg=4326)
                    GeoJson(
                        geom_gdf.geometry,
                        style_function=lambda x, c=color_map.get(row["sensor"].lower(), "gray"):
                            {"color": c, "fill": False, "weight": 2, "opacity": 0.5},
                        tooltip=f"{row['sensor']} {row['year']} {row['suffix']}"
                    ).add_to(m)
            except Exception:
                continue
    
    # Fit map to AOI bounds
    minx, miny, maxx, maxy = aoi_gdf.total_bounds
    m.fit_bounds([[miny, minx], [maxy, maxx]])
    return m

# Generate interactive map
m = plot_aoi_interactive_sensors(scene_df, AOI_PATH)
map_path = Path(RAW_DIR) / "aoi_scene_map.html"
m.save(str(map_path))
print(f"Interactive map saved to {map_path}")